# Patient-Level PD-L1 Expression and PET/CT Metabolic Measurements in NSCLC Exploration with `mlcroissant`
This notebook provides a guided walkthrough for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

> **Dataset Title:** Patient-Level PD-L1 Expression and PET/CT Metabolic Measurements in Non-Small Cell Lung Cancer, January 2021–January 2024

> **Identifier:** 10.71728/senscience.vcnn-tj27

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcnn-tj27/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Version:", metadata.version)
print("License:", metadata.license)
print("Keywords:", metadata.keywords)
print("Identifier:", metadata.identifier)

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id`.

This step inspects the record sets in the dataset. It prints their `@id`, names, and associated fields/columns.

In [ ]:
# List all record sets in the dataset and their fields
record_sets = list(dataset.record_sets())  # Returns a list of CroissantRecordSet
record_sets_ids = []

print("Record Sets Overview:")
for rs in record_sets:
    print("- RecordSet @id:", rs.id)
    print("  Name:", rs.name)
    print("  Description:", rs.description)
    print("  Fields:")
    for field in rs.fields:
        print("    * Field @id:", field.id, "/ Column:", field.column, "/ Name:", field.name)
    record_sets_ids.append(rs.id)
    print()

# Show example records from the first record set
if record_sets:
    first_rs_id = record_sets[0].id
    print(f"Records from RecordSet @id: {first_rs_id}")
    for i, record in enumerate(dataset.records(record_set=first_rs_id)):
        print(record)
        if i >= 2:  # Show only first 3 records
            break

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis.

**Note:** Each DataFrame is keyed by the record set `@id`.

In [ ]:
# Extracting records from all record sets
dataframes = {}

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df

# Preview columns of the main record set
main_record_set_id = record_sets[0].id if record_sets else None
if main_record_set_id:
    print(f"Columns for RecordSet @id: {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())

    print("Data preview:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using column `@id`s as references.

We'll select numeric fields such as 'Lesion SUVmax', filter by threshold, normalize, and group by attributes like 'histological classification' or 'PD-L1 expression'.

In [ ]:
# Find a numeric field for analysis in the main record set
rs_obj = next((rs for rs in record_sets if rs.id == main_record_set_id), None)
numeric_field_id = None

for field in rs_obj.fields:
    if field.data_type in ("Float", "Integer"):
        numeric_field_id = field.id
        print(f"Selected numeric field for EDA: {numeric_field_id} ({field.name})")
        break

if numeric_field_id:
    df = dataframes[main_record_set_id]
    # Filter records based on threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize numeric field
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, normalized_col]].head())

    # Find a categorical/group field
    group_field_id = None
    for field in rs_obj.fields:
        if field.data_type == "Text" and field.id != numeric_field_id:
            group_field_id = field.id
            print(f"Grouping by field @id: {group_field_id} ({field.name})")
            break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
We'll use matplotlib and seaborn to plot distributions for numeric and grouped fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if numeric_field_id and main_record_set_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id], bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouped_df exists, visualize group means
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 6))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded FAIR^2 NSCLC patient-level dataset using the Croissant schema and `mlcroissant`.
- Inspected and extracted record sets by their `@id` identifiers; fields and columns consistently referenced by `@id`.
- Performed simple filtering, normalization, and grouping on a numeric field, visualizing distributions and group means.
- Dataset supports research into tumor biology, biomarker stratification, and clinical correlations in NSCLC, but is not suitable for direct clinical use.